# 📦 Notebook 1: Inventory Management Across Micro-Fulfillment Centers

Gopuff doesn’t use third-party stores. It owns **hundreds of small warehouses** (micro-fulfillment centers, or DCs) spread across a city. When a customer opens the app, the system must instantly show what’s available — by combining inventory from every DC that can deliver to them.

This notebook covers:
1. How inventory is structured across multiple DCs
2. Querying aggregated availability for a customer location
3. Placing orders with **atomic transactions** (no double-booking)
4. Speeding up reads with **Redis cache-aside**

## Learning Goals

- Understand the difference between an **Item** (catalog entry) and **Inventory** (physical stock at a DC)
- Write SQL that aggregates inventory across multiple warehouses
- Use Postgres `SERIALIZABLE` transactions to prevent two customers from buying the same last item
- Implement cache-aside with Redis to meet the < 100 ms latency requirement

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/gopuff
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, DB `gopuff`
- **RedisInsight** (Redis GUI): http://localhost:5540
  Add database → Host `redis`, Port `6379`

### Kernel Selection (VS Code)
Select the `.venv` kernel from the kernel picker (top-right of the notebook).
If it doesn’t appear, reload the VS Code window (`Cmd+Shift+P` → “Reload Window”).

In [1]:
import psycopg2
import psycopg2.extras
import redis
import json
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "gopuff",
    "user": "demo",
    "password": "demo",
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True,
}


def get_db_connection():
    """Open a new Postgres connection."""
    return psycopg2.connect(**DB_CONFIG)


def get_redis_client():
    """Return a Redis client."""
    return redis.Redis(**REDIS_CONFIG)


conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM distribution_centers")
dc_count = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM items")
item_count = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM inventory")
inv_count = cur.fetchone()[0]
conn.close()

r = get_redis_client()
r.ping()

print(f"✅ Connected! {dc_count} DCs, {item_count} items, {inv_count} inventory rows")
print(f"✅ Redis is up")

✅ Connected! 8 DCs, 30 items, 187 inventory rows
✅ Redis is up


## 1️⃣ Understanding the Data Model

Think of it like a convenience store chain:

- **Item** = “Coca-Cola 12-pack” (the *type* of product in the catalog)
- **Inventory** = “DC Downtown has 45 Coca-Cola 12-packs” (the *physical count* at a location)
- **Distribution Center** = a small warehouse at a specific lat/lon

A single item can exist in many DCs. The customer sees the *total* across all nearby DCs.

In [2]:
conn = get_db_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT id, name, city, latitude, longitude, capacity_sqft
    FROM distribution_centers
    ORDER BY id
""")

print("🏭 Distribution Centers")
print("=" * 75)
for dc in cur.fetchall():
    print(f"  DC {dc['id']:2d} | {dc['name']:<18s} | {dc['city']:<12s} | "
          f"({dc['latitude']:.4f}, {dc['longitude']:.4f}) | {dc['capacity_sqft']} sqft")

conn.close()

🏭 Distribution Centers
  DC  1 | DC Downtown        | Austin       | (30.2672, -97.7431) | 5000 sqft
  DC  2 | DC East            | Austin       | (30.2634, -97.6934) | 4000 sqft
  DC  3 | DC South           | Austin       | (30.2060, -97.7954) | 6000 sqft
  DC  4 | DC North           | Austin       | (30.3716, -97.7064) | 4500 sqft
  DC  5 | DC West            | Austin       | (30.2650, -97.8500) | 3500 sqft
  DC  6 | DC University      | Austin       | (30.2849, -97.7341) | 3000 sqft
  DC  7 | DC Airport         | Austin       | (30.1975, -97.6664) | 5500 sqft
  DC  8 | DC Round Rock      | Round Rock   | (30.5083, -97.6789) | 4000 sqft


In [3]:
conn = get_db_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT
        dc.name AS dc_name,
        i.name  AS item_name,
        inv.quantity,
        inv.reorder_point
    FROM inventory inv
    JOIN distribution_centers dc ON dc.id = inv.dc_id
    JOIN items i ON i.id = inv.item_id
    WHERE i.name = 'Coca-Cola 12-pack'
    ORDER BY inv.quantity DESC
""")

rows = cur.fetchall()
total = sum(r['quantity'] for r in rows)

print("🥤 Coca-Cola 12-pack — Inventory by DC")
print("=" * 50)
for r in rows:
    bar = "█" * (r['quantity'] // 2)
    print(f"  {r['dc_name']:<18s} | {r['quantity']:4d} units {bar}")
print(f"{'':>18s}   ------")
print(f"{'Total':>18s} | {total:4d} units")

conn.close()

🥤 Coca-Cola 12-pack — Inventory by DC
  DC South           |   92 units ██████████████████████████████████████████████
  DC East            |   87 units ███████████████████████████████████████████
  DC North           |   67 units █████████████████████████████████
  DC Airport         |   59 units █████████████████████████████
  DC Downtown        |   50 units █████████████████████████
  DC West            |   32 units ████████████████
  DC Round Rock      |   28 units ██████████████
                     ------
             Total |  415 units


## 2️⃣ Aggregated Availability Query

When a customer opens the app, we need to:
1. Find which DCs can deliver to their location (we’ll cover this in Notebook 2)
2. Sum up inventory across those DCs for every item

For now, let’s assume we already know the nearby DC IDs and focus on the aggregation query.

**Key insight**: The customer doesn’t care *which* warehouse the item comes from. They just want to know “is Coke available and how many?”

In [4]:
def get_availability(dc_ids):
    """
    Given a list of nearby DC IDs, return aggregated availability.
    Each item shows the total quantity across all the given DCs.
    """
    conn = get_db_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT
            i.id        AS item_id,
            i.name      AS item_name,
            i.category,
            i.base_price,
            SUM(inv.quantity) AS total_quantity
        FROM inventory inv
        JOIN items i ON i.id = inv.item_id
        WHERE inv.dc_id = ANY(%s)
          AND inv.quantity > 0
        GROUP BY i.id, i.name, i.category, i.base_price
        ORDER BY i.category, i.name
    """, (dc_ids,))

    results = cur.fetchall()
    conn.close()
    return results


nearby_dcs = [1, 2, 6]
availability = get_availability(nearby_dcs)

print(f"📋 Availability for DCs {nearby_dcs}")
print(f"   ({len(availability)} items in stock)\n")

current_cat = None
for item in availability:
    if item['category'] != current_cat:
        current_cat = item['category']
        print(f"\n  ── {current_cat} ──")
    print(f"    {item['item_name']:<28s} ${float(item['base_price']):>6.2f}  "
          f"({item['total_quantity']} in stock)")

📋 Availability for DCs [1, 2, 6]
   (28 items in stock)


  ── Baby ──
    Baby Wipes 72ct              $  3.49  (52 in stock)
    Diapers Size 3 27ct          $ 12.49  (74 in stock)

  ── Beverages ──
    Coca-Cola 12-pack            $  6.99  (137 in stock)
    Dasani Water 24pk            $  5.99  (35 in stock)
    Gatorade Blue                $  2.29  (166 in stock)
    Red Bull 4-pack              $  8.49  (119 in stock)

  ── Dairy ──
    Cheddar Cheese Block         $  4.29  (42 in stock)
    Eggs Large 12ct              $  3.99  (42 in stock)
    Greek Yogurt Vanilla         $  5.99  (23 in stock)
    Whole Milk Gallon            $  4.49  (51 in stock)

  ── Electronics ──
    Phone Charger USB-C          $ 12.99  (97 in stock)

  ── Frozen ──
    Frozen Burritos 8pk          $  6.49  (119 in stock)
    Frozen Pizza Pepperoni       $  7.49  (104 in stock)
    Ice Cream Vanilla            $  5.99  (45 in stock)

  ── Health ──
    Band-Aid Variety             $  4.49  (68 in stoc

## 3️⃣ The Naive Order Path (⚠️ Broken Under Concurrency)

Before we write the "right" version, let's write the version a junior engineer might write first — no transaction, just "read the quantity, subtract, write it back." This is **bad practice** on purpose, and we'll prove it breaks.

```
Thread A:           Thread B:
  read qty = 1
                      read qty = 1   ← both see stock!
  write qty = 0
                      write qty = 0  ← should be -1, we just oversold
```

The bug is called a **lost update** / **double-booking**. It happens any time two requests read the same row, compute something, and write back without coordination.


In [5]:
# ⚠️ DO NOT USE IN PRODUCTION — this is the "bad" version on purpose.
import threading

def place_order_naive(customer_id, dc_id, item_id, quantity, barrier=None):
    """
    Naive order: read quantity, check, then decrement.
    No transaction. No row lock. Vulnerable to race conditions.
    Each call uses its own connection (same as the fixed version).
    """
    conn = get_db_connection()
    cur = conn.cursor()

    # Step 1: read current quantity
    cur.execute(
        "SELECT quantity FROM inventory WHERE dc_id = %s AND item_id = %s",
        (dc_id, item_id),
    )
    row = cur.fetchone()
    current_qty = row[0] if row else 0

    # Barrier forces both threads to reach here *before* either writes.
    # In the real world the interleaving is random — we just make it reproducible.
    if barrier is not None:
        barrier.wait()

    if current_qty < quantity:
        conn.close()
        return {"customer": customer_id, "success": False, "reason": "out of stock"}

    # Step 2: decrement (unconditionally — the bug!)
    cur.execute(
        "UPDATE inventory SET quantity = quantity - %s WHERE dc_id = %s AND item_id = %s",
        (quantity, dc_id, item_id),
    )
    conn.commit()
    conn.close()
    return {"customer": customer_id, "success": True, "took": current_qty}


# Reset the demo item to exactly 1 unit
conn = get_db_connection()
cur = conn.cursor()
cur.execute("UPDATE inventory SET quantity = 1 WHERE dc_id = 1 AND item_id = 17")
conn.commit()
conn.close()

# Two threads, one barrier, one item — classic oversell setup
barrier = threading.Barrier(2)
results = {}

def buyer(name, cid):
    results[name] = place_order_naive(cid, dc_id=1, item_id=17, quantity=1, barrier=barrier)

t1 = threading.Thread(target=buyer, args=("Alice", 100))
t2 = threading.Thread(target=buyer, args=("Bob", 200))
t1.start(); t2.start(); t1.join(); t2.join()

conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT quantity FROM inventory WHERE dc_id = 1 AND item_id = 17")
final_qty = cur.fetchone()[0]
conn.close()

for name, res in results.items():
    print(f"  {name}: {res}")
print(f"\n💥 Final inventory: {final_qty}  (negative = we sold something we didn't have)")
print("   This is the bug we need to fix.")


  Alice: {'customer': 100, 'success': True, 'took': 1}
  Bob: {'customer': 200, 'success': True, 'took': 1}

💥 Final inventory: -1  (negative = we sold something we didn't have)
   This is the bug we need to fix.


## 4️⃣ The Fix — Atomic Transactions

### The Double-Booking Problem

Imagine there’s **1 avocado pack left** at DC Downtown. Two customers tap “Buy” at the same time:

```
Customer A: reads quantity = 1  ✓ available!
Customer B: reads quantity = 1  ✓ available!
Customer A: decrements to 0, creates order
Customer B: decrements to -1 ← BUG! We promised something we don’t have
```

### The Fix: Postgres Transactions

We wrap the entire check-and-decrement in a **single database transaction** with `SERIALIZABLE` isolation. Postgres guarantees that if two transactions conflict, one of them will be rolled back.

Think of it like a lock on a fitting room door — only one person can try on the last pair of jeans at a time.

In [6]:
def place_order(customer_id, dc_id, items_to_order,
                delivery_address, delivery_lat, delivery_lon):
    """
    Place an order atomically.
    items_to_order: [{"item_id": 1, "quantity": 2}, ...]

    Steps inside one transaction:
      1. Check inventory for every item
      2. If anything is out of stock -> roll back
      3. Decrement inventory
      4. Create the order + order items
      5. Commit
    """
    conn = get_db_connection()
    conn.set_session(isolation_level="SERIALIZABLE")
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        total_price = 0

        for entry in items_to_order:
            cur.execute("""
                SELECT inv.quantity, i.base_price
                FROM inventory inv
                JOIN items i ON i.id = inv.item_id
                WHERE inv.dc_id = %s AND inv.item_id = %s
                FOR UPDATE
            """, (dc_id, entry["item_id"]))

            row = cur.fetchone()
            if row is None or row["quantity"] < entry["quantity"]:
                conn.rollback()
                available = row["quantity"] if row else 0
                return {
                    "success": False,
                    "error": f"Item {entry['item_id']} — requested {entry['quantity']}, "
                             f"only {available} available at DC {dc_id}"
                }
            total_price += float(row["base_price"]) * entry["quantity"]

        for entry in items_to_order:
            cur.execute("""
                UPDATE inventory
                SET quantity = quantity - %s, updated_at = NOW()
                WHERE dc_id = %s AND item_id = %s
            """, (entry["quantity"], dc_id, entry["item_id"]))

        cur.execute("""
            INSERT INTO orders (customer_id, dc_id, status, total_price,
                                delivery_address, delivery_lat, delivery_lon)
            VALUES (%s, %s, 'confirmed', %s, %s, %s, %s)
            RETURNING id
        """, (customer_id, dc_id, total_price, delivery_address,
              delivery_lat, delivery_lon))
        order_id = cur.fetchone()["id"]

        for entry in items_to_order:
            cur.execute("SELECT base_price FROM items WHERE id = %s", (entry["item_id"],))
            price = cur.fetchone()["base_price"]
            cur.execute("""
                INSERT INTO order_items (order_id, item_id, quantity, unit_price)
                VALUES (%s, %s, %s, %s)
            """, (order_id, entry["item_id"], entry["quantity"], price))

        conn.commit()
        return {"success": True, "order_id": order_id, "total_price": total_price}

    except psycopg2.errors.SerializationFailure:
        conn.rollback()
        return {"success": False, "error": "Transaction conflict — please retry"}
    finally:
        conn.close()


result = place_order(
    customer_id=42, dc_id=1,
    items_to_order=[{"item_id": 1, "quantity": 2}, {"item_id": 9, "quantity": 1}],
    delivery_address="123 Main St, Austin, TX",
    delivery_lat=30.2672, delivery_lon=-97.7431,
)

if result["success"]:
    print(f"✅ Order #{result['order_id']} placed! Total: ${result['total_price']:.2f}")
else:
    print(f"❌ Order failed: {result['error']}")

❌ Order failed: Item 9 — requested 1, only 0 available at DC 1


### 🧪 Simulating the Double-Booking Race Condition

Let’s prove the transaction works. We’ll set an item to quantity = 1, then try to buy it from two “customers” at the same time using threads.

In [7]:
import threading

conn = get_db_connection()
cur = conn.cursor()
cur.execute("UPDATE inventory SET quantity = 1 WHERE dc_id = 1 AND item_id = 17")
conn.commit()
conn.close()
print("Setup: DC 1 now has exactly 1 avocado pack")

results = {}

def try_order(customer_name, customer_id):
    result = place_order(
        customer_id=customer_id, dc_id=1,
        items_to_order=[{"item_id": 17, "quantity": 1}],
        delivery_address=f"{customer_name}'s house",
        delivery_lat=30.27, delivery_lon=-97.74,
    )
    results[customer_name] = result

t1 = threading.Thread(target=try_order, args=("Alice", 100))
t2 = threading.Thread(target=try_order, args=("Bob", 200))
t1.start()
t2.start()
t1.join()
t2.join()

print("\n🏁 Race Condition Results:")
for name, res in results.items():
    status = "✅ Got it!" if res["success"] else f"❌ {res['error']}"
    print(f"  {name}: {status}")

conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT quantity FROM inventory WHERE dc_id = 1 AND item_id = 17")
final_qty = cur.fetchone()[0]
conn.close()
print(f"\n📊 Final inventory: {final_qty} (should be 0, never negative)")

Setup: DC 1 now has exactly 1 avocado pack

🏁 Race Condition Results:
  Alice: ✅ Got it!
  Bob: ❌ Transaction conflict — please retry



📊 Final inventory: 0 (should be 0, never negative)


## 5️⃣ Speeding Up Reads with Redis (Cache-Aside)

The interview requires availability queries under **100 ms**. With 20k queries/second (estimated from 10M orders/day), hitting Postgres every time is too expensive.

### Cache-Aside Pattern

```
Customer Request
       │
       ▼
  ┌─ Check Redis ─┐
  │               │
  │  HIT?  ───── Return cached result (< 5 ms)
  │               │
  │  MISS? ───── Query Postgres
  │               │
  │               ├── Store in Redis (TTL = 60s)
  │               │
  │               └── Return result
  └───────────────┘
```

**Why a 60-second TTL?** Inventory changes when orders are placed, but a small delay is acceptable for *browsing*. When someone actually places an order, we check the real database (strong consistency for writes, eventual consistency for reads).

In [8]:
CACHE_TTL_SECONDS = 60

def get_availability_cached(dc_ids):
    """
    Cache-aside availability lookup.
    Returns (items, source) where source is 'cache' or 'database'.
    """
    r = get_redis_client()
    cache_key = "availability:" + ",".join(str(d) for d in sorted(dc_ids))

    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), "cache"

    items = get_availability(dc_ids)
    serializable = [
        {k: (float(v) if hasattr(v, 'as_tuple') else v) for k, v in row.items()}
        for row in items
    ]
    r.setex(cache_key, CACHE_TTL_SECONDS, json.dumps(serializable))
    return serializable, "database"


r = get_redis_client()
r.flushdb()

start = time.time()
items, source = get_availability_cached([1, 2, 6])
first_ms = (time.time() - start) * 1000
print(f"1st call: {source:>8s} | {first_ms:.1f} ms | {len(items)} items")

start = time.time()
items, source = get_availability_cached([1, 2, 6])
second_ms = (time.time() - start) * 1000
print(f"2nd call: {source:>8s} | {second_ms:.1f} ms | {len(items)} items")

print(f"\n⚡ Cache speedup: {first_ms / max(second_ms, 0.01):.1f}x faster")

1st call: database | 21.8 ms | 28 items
2nd call:    cache | 3.3 ms | 28 items

⚡ Cache speedup: 6.7x faster


### Cache Invalidation on Order

When an order is placed, the cached availability is **stale** (it still shows the old quantity). We should invalidate the relevant cache entries so the next read fetches fresh data.

In [9]:
def invalidate_availability_cache(dc_id):
    r = get_redis_client()
    pattern = "availability:*"
    deleted = 0
    for key in r.scan_iter(pattern):
        dc_ids_in_key = key.split(":")[1].split(",")
        if str(dc_id) in dc_ids_in_key:
            r.delete(key)
            deleted += 1
    return deleted


items, source = get_availability_cached([1, 2, 6])
print(f"Before order: source = {source}")

result = place_order(
    customer_id=99, dc_id=1,
    items_to_order=[{"item_id": 1, "quantity": 1}],
    delivery_address="456 Oak Ave", delivery_lat=30.27, delivery_lon=-97.74,
)
print(f"Order result: {'✅' if result['success'] else '❌'}")

deleted = invalidate_availability_cache(dc_id=1)
print(f"Invalidated {deleted} cache key(s)")

items, source = get_availability_cached([1, 2, 6])
print(f"After invalidation: source = {source}  (fresh data!)")

Before order: source = cache
Order result: ✅
Invalidated 1 cache key(s)


After invalidation: source = database  (fresh data!)


## 📊 Performance Comparison

Let’s measure many reads with and without caching to see the difference at scale.

In [10]:
import statistics

def benchmark(fn, label, iterations=50):
    times = []
    for _ in range(iterations):
        start = time.time()
        fn()
        times.append((time.time() - start) * 1000)

    print(f"\n📈 {label} ({iterations} iterations)")
    print(f"   Avg: {statistics.mean(times):.1f} ms")
    print(f"   Med: {statistics.median(times):.1f} ms")
    print(f"   P95: {sorted(times)[int(iterations * 0.95)]:.1f} ms")
    print(f"   Min: {min(times):.1f} ms | Max: {max(times):.1f} ms")
    return times

dc_ids = [1, 2, 6]
db_times = benchmark(lambda: get_availability(dc_ids), "Direct Postgres")

r = get_redis_client()
r.flushdb()
get_availability_cached(dc_ids)
cache_times = benchmark(lambda: get_availability_cached(dc_ids), "Redis Cache-Aside")

print(f"\n⚡ Average speedup: {statistics.mean(db_times) / statistics.mean(cache_times):.1f}x")


📈 Direct Postgres (50 iterations)
   Avg: 18.8 ms
   Med: 18.1 ms
   P95: 24.8 ms
   Min: 14.4 ms | Max: 35.1 ms

📈 Redis Cache-Aside (50 iterations)
   Avg: 2.7 ms
   Med: 2.5 ms
   P95: 4.0 ms
   Min: 1.8 ms | Max: 4.2 ms

⚡ Average speedup: 6.9x


## 6️⃣ Retrying on Serialization Failures

`SERIALIZABLE` isolation isn't free — when two transactions conflict, Postgres **aborts one of them** with a `SerializationFailure`. In a real service you want to automatically retry instead of failing the user request.

This is the standard pattern used by any system that leans on serializable transactions (Postgres, CockroachDB, Spanner…): wrap the call in a small retry loop with backoff.


In [11]:
import random

def place_order_with_retry(max_retries=3, **kwargs):
    """
    Retry place_order on serialization failures with exponential backoff + jitter.
    Real-world services usually cap retries at 3–5 and give up with a 503 after that.
    """
    for attempt in range(max_retries):
        result = place_order(**kwargs)
        # place_order already swallows SerializationFailure and returns an error dict,
        # so we check the error message to decide whether to retry.
        if result.get("success"):
            return {**result, "attempts": attempt + 1}
        if "conflict" not in result.get("error", "").lower():
            # Business failure (e.g. out of stock) — don't retry
            return {**result, "attempts": attempt + 1}
        # Exponential backoff with jitter: 20ms, 40ms, 80ms, …
        sleep_ms = (2 ** attempt) * 20 + random.randint(0, 10)
        time.sleep(sleep_ms / 1000)
    return {"success": False, "error": "max retries exceeded", "attempts": max_retries}


# Quick sanity run
result = place_order_with_retry(
    max_retries=3,
    customer_id=7, dc_id=1,
    items_to_order=[{"item_id": 1, "quantity": 1}],
    delivery_address="1 Retry Lane",
    delivery_lat=30.27, delivery_lon=-97.74,
)
print(result)


{'success': True, 'order_id': 505, 'total_price': 6.99, 'attempts': 1}


### 📝 Real-World Note: Inventory Reservations

In production, Gopuff-style services don't decrement stock at "Place Order." They **reserve** it when the customer enters checkout (e.g., hold for 5 minutes), then commit the decrement at payment success or release it on timeout / cancel.

The schema extension is typically:

```sql
CREATE TABLE inventory_holds (
    id SERIAL PRIMARY KEY,
    dc_id INT, item_id INT, quantity INT,
    customer_id INT,
    expires_at TIMESTAMP   -- hold auto-releases if payment stalls
);
```

The availability query then becomes `quantity - SUM(active_holds)`. We stop here in this notebook, but interviewers love this detail.


## 🔑 Key Takeaways

| Concept | What We Did |
|---------|-------------|
| **Item vs Inventory** | Items are catalog entries; Inventory is physical stock at a DC |
| **Aggregated availability** | `SUM(quantity)` across nearby DCs — customer sees one number |
| **Atomic ordering** | `SERIALIZABLE` transaction + `FOR UPDATE` row lock prevents double-booking |
| **Cache-aside** | Check Redis first → miss → query Postgres → store in Redis with TTL |
| **Cache invalidation** | Delete stale cache keys after an order changes inventory |

### Interview Tip

When asked about Gopuff, start with the **data model** (Item vs Inventory distinction), then explain **read path** (cache-aside for browsing) vs **write path** (serializable transactions for ordering). This shows you understand the different consistency requirements for reads and writes.

## 🧹 Cleanup

In [12]:
r = get_redis_client()
keys = r.keys("availability:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

🧹 Cleaned up 1 Redis keys
